# День 4 — Первая классификация

## Цель
Обучить **первую классификационную модель**, сравнить её с **baseline** из дня 3 и понять, что означают метрики.

## fit / predict — как scikit-learn обучает модель

В scikit-learn у всех моделей один и тот же интерфейс:

1. **`fit(X_train, y_train)`** — модель смотрит на примеры и подстраивает параметры.
2. **`predict(X_test)`** — модель выдаёт предсказанные классы для новых строк.

```python
model = LogisticRegression()
model.fit(X_train, y_train)      # учимся только на train
y_pred = model.predict(X_test)   # проверяем на test
```

Правило из day 2–3: **fit — только train**, метрики — на **test**.

## Метрики: accuracy не всегда достаточно

| Метрика | Что значит |
|---------|------------|
| **accuracy** | доля правильных ответов |
| **precision** | из предсказанных «да» — сколько реально «да» |
| **recall** | из реальных «да» — сколько модель нашла |
| **F1** | баланс precision и recall |

При **дисбалансе классов** (например, 95% спама и 5% не-спама) accuracy может быть высокой, если модель всегда говорит «спам». Тогда смотрят **precision / recall / F1** по каждому классу.

На Iris классы сбалансированы — accuracy здесь уместна, но `classification_report` всё равно полезен: видно, какой класс модель путает.

## Confusion matrix (матрица ошибок)

`confusion_matrix(y_test, y_pred)` показывает, **где модель ошибается**:

- строки — **истинный** класс;
- столбцы — **предсказанный** класс;
- диагональ — правильные ответы;
- вне диагонали — путаница между классами.

Пример: если versicolor часто предсказывается как virginica — это видно в матрице, даже когда accuracy высокая.

## Сравнение с baseline

Высокая accuracy сама по себе ничего не значит, если baseline уже близок к ней.

Порядок дня:

```
split → baseline (day 3) → модель 1 → модель 2 → сравнить метрики
```

Модель **обязана** побить baseline. Если нет — признаки не используются или модель настроена неверно.

## Задания

1. Тот же датасет Iris и split, что в day 2–3.
2. Обучить `LogisticRegression` и `DecisionTreeClassifier`.
3. `predict` на test → `accuracy`, `classification_report`, `confusion_matrix`.
4. Сравнить модели между собой и с baseline.
5. Написать 6–8 строк выводов.

In [ ]:
# 1. Загрузка Iris и train/test split (как в day 2–3)

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

dataset = load_iris(as_frame=True)
X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Классы:", list(dataset.target_names))

In [ ]:
# 2. Baseline из day 3 (для сравнения)

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

y_pred_baseline = baseline.predict(X_test)
acc_baseline = accuracy_score(y_test, y_pred_baseline)
print(f"Baseline accuracy: {acc_baseline:.2f}")

In [ ]:
# 3. LogisticRegression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

lr = LogisticRegression(max_iter=200, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)

print(f"LogisticRegression accuracy: {acc_lr:.2f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred_lr, target_names=dataset.target_names))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_lr))

In [ ]:
# 4. DecisionTreeClassifier

from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)

print(f"DecisionTree accuracy: {acc_dt:.2f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred_dt, target_names=dataset.target_names))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_dt))

In [ ]:
# 5. Сравнение моделей

import pandas as pd

comparison = pd.DataFrame({
    "model": ["baseline", "LogisticRegression", "DecisionTree"],
    "accuracy": [acc_baseline, acc_lr, acc_dt],
})
comparison["vs_baseline"] = comparison["accuracy"] - acc_baseline
comparison

## Выводы (6–8 строк)

1. **LogisticRegression** чуть лучше на test: accuracy **0.97** против **0.93** у дерева и **0.33** у baseline.
2. Обе ML-модели **сильно** бьют baseline — признаки Iris действительно предсказуемы (как ожидали после EDA в day 3).
3. **Setosa** предсказывается без ошибок у обеих моделей — класс хорошо отделён по признакам лепестков.
4. Ошибки у обеих моделей — путаница **versicolor ↔ virginica** (1 образец в confusion matrix).
5. У LogisticRegression versicolor: recall 0.90 (один versicolor назван virginica). У дерева — симметричная путаница в обе стороны.
6. **Accuracy** здесь достаточна (классы сбалансированы), но `classification_report` показывает, *где* именно модель ошибается.
7. Результатам на test **можно доверять**: split честный, метрики считались только на невиденных данных.
8. На 30 test-образцах разница 0.97 vs 0.93 мала — для уверенного выбора модели понадобится CV (day 6).

---

# Day 4 — First Classification

## Goal
Train the **first classification model**, compare it with the **baseline** from day 3, and understand what the metrics mean.

## fit / predict — how scikit-learn trains a model

All scikit-learn models share the same interface:

1. **`fit(X_train, y_train)`** — the model learns from examples.
2. **`predict(X_test)`** — the model outputs predicted classes for new rows.

Rule from days 2–3: **fit on train only**, metrics on **test**.

## Metrics: accuracy is not always enough

| Metric | Meaning |
|--------|---------|
| **accuracy** | share of correct predictions |
| **precision** | of predicted positives, how many are truly positive |
| **recall** | of true positives, how many the model found |
| **F1** | balance of precision and recall |

With **class imbalance**, accuracy can be misleading. Use **precision / recall / F1** per class.

On Iris classes are balanced — accuracy is fine, but `classification_report` still shows which class the model confuses.

## Confusion matrix

`confusion_matrix(y_test, y_pred)` shows **where the model makes mistakes**:

- rows — **true** class;
- columns — **predicted** class;
- diagonal — correct predictions;
- off-diagonal — confusion between classes.

## Compare with baseline

High accuracy alone means little if the baseline is already close.

```
split → baseline (day 3) → model 1 → model 2 → compare metrics
```

The model **must** beat the baseline.

## Tasks (do by hand)

1. Notebook: `week-3/notebooks/day04_classification.ipynb`.
2. On Iris (or Breast Cancer), train `LogisticRegression` and `DecisionTreeClassifier`.
3. `predict` on test → `accuracy`, `classification_report`, `confusion_matrix`.
4. Compare models with each other and with the day 3 baseline.
5. Write 6–8 lines of conclusions: which model is better, where it fails, can you trust the result.

## Conclusions (6–8 lines)

1. **LogisticRegression** is slightly better on test: accuracy **0.97** vs **0.93** for the tree and **0.33** for baseline.
2. Both ML models **strongly** beat baseline — Iris features are predictable (as expected after day 3 EDA).
3. **Setosa** is predicted perfectly by both models.
4. Errors are **versicolor ↔ virginica** confusion (1 sample in the confusion matrix).
5. LogisticRegression: versicolor recall 0.90. The tree has symmetric confusion both ways.
6. **Accuracy** is enough here (balanced classes), but `classification_report` shows *where* the model fails.
7. Test results are **trustworthy**: honest split, metrics on unseen data only.
8. On 30 test samples, 0.97 vs 0.93 is a small gap — CV (day 6) will help choose confidently.